# 05 LBBD —— 逻辑割（no-good / Hooker）的推导与有效性

## 主问题（CP-SAT 分配）与子问题（单车辆 TSP-TW）

$$\min\theta\ \text{s.t.}\ \sum_k a_{ik}=1,\ a_{ik}\le v_k,\ \sum_i\delta_i a_{ik}\le Q,\ \sum_k v_k=K^*,\ \theta\le\overline{UB}-1$$

子问题对车辆 $k$ 的客户集 $S_k$ 解单车辆 TSP-TW（min 距离），返回可行性与成本 $c_k$。
逻辑 Benders 不需要对偶——**割由子问题的可行性/最优性直接导出**。

## 割一：no-good（不可行集）

**事实**：若 $S_k$ 不可行（无满足时间窗+容量+返回仓库的排序），则任何把 $S_k$ 整组分配给车 $k$ 的
分配都不可行。**割**：$\sum_{i\in S_k}(1-a_{ik})\ge 1$（至少移走一个客户）。

**加强**：贪心收缩到**最小不可行核心**（逐客户尝试移除，仍不可行则移除），核心更小 ⇒ 割更强。
**有效性**：不切掉任何可行分配（只禁止了必然不可行的分配）。

## 割二：Hooker 最优性割（可行集）

设车辆 $k$ 对 $S_k$ 的最优成本为 $c_k$，定义**单客户移除节省**
$\delta_{ik}=c_k-c_k(S_k\setminus\{i\})$（子问题重解精确得到；移除后不可行时取 $\delta_{ik}=c_k$）。
对任意分配 $a'$（车 $k$ 分到 $S'_k$），在"多客户移除节省可加"的标准假设下：

$$c_k(S'_k)\ \ge\ c_k-\sum_{i\in S_k\setminus S'_k}\delta_{ik} \qquad\Longrightarrow\qquad
\theta\ \ge\ \sum_k\Big[c_k-\sum_{i\in S_k}\delta_{ik}(1-a_{ik})\Big]$$

**有效性口径**：单客户移除是精确的（重解验证）；多客户同时移除依赖标准可加性假设（TSP-TW 子问题的
经典 LBBD 口径，02/03/05 notebook 已如实声明，最终结论与 01/02/03/04/07 交叉验证一致）。

## 割三：精确成本界（严格有效）

$$\theta\ \ge\ C\cdot\Big(1-\sum_k\sum_{i\in S_k}(1-a_{ik})\Big)$$

指示器 $\mathbb 1[a=a^*]=1-\sum(1-a)\in\{0,1\}$：完全复现该分配时右端=$C$（精确），否则右端≤0≤θ（平凡成立）。
**无条件有效**，是主问题不可行性证明的"锚"。

## 最优性证明

主问题含 $\theta\le\overline{UB}-1$（必须找到比现任解更好的分配）。若主问题**不可行**，则任何可行分配
都被割覆盖且成本 ≥ UB ⇒ **UB 最优**。这就是 05 第 22 轮"主问题不可行 ⇒ 已证明最优"的数学依据。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


from lbbd import solve_route, shrink
# no-good 演示：全部 25 客户塞一车 -> 不可行 -> 收缩核心
S_all = tuple(range(1, n+1))
st, route, cost = solve_route(S_all)
print(f"全部客户一车: {st} | 最小不可行核心 = {shrink(S_all)}")
print("no-good 割: Σ_{i∈核心}(1-a_i) >= 1（至少移走一个客户）")

# Hooker 割演示：最优路线集 S1 的成本与移除节省
S1 = ROUTES[0]
st1, r1, ck = solve_route(S1)
print()
print(f"路线 {S1}: 最优成本 c_k = {round(ck, 4)} | 状态 {st1}")
deltas = {}
for i in S1:
    S2 = tuple(j for j in S1 if j != i)
    st3, _, c2 = solve_route(S2)
    dlt = ck - c2 if st3 == "OPTIMAL" else ck
    deltas[i] = round(dlt, 4)
print("单客户移除节省 δ:", deltas)
print("Hooker 割: theta >= c_k - Σ δ_i(1-a_i)（θ 为该车成本的下界近似）")

# 精确界割（严格有效）
print("精确界割: theta >= C·(1 - Σ(1-a_i))（仅完全复现该分配时抬到 C，否则 0）")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）


全部客户一车: INFEASIBLE | 最小不可行核心 = (18, 25)
no-good 割: Σ_{i∈核心}(1-a_i) >= 1（至少移走一个客户）

路线 (13, 17, 18, 19, 15, 16, 14, 12): 最优成本 c_k = 95.8847 | 状态 OPTIMAL
单客户移除节省 δ: {13: 1.5042, 17: 0.0, 18: 2.169, 19: 2.9289, 15: 2.9289, 16: 1.6148, 14: 0.0, 12: 1.7215}
Hooker 割: theta >= c_k - Σ δ_i(1-a_i)（θ 为该车成本的下界近似）
精确界割: theta >= C·(1 - Σ(1-a_i))（仅完全复现该分配时抬到 C，否则 0）
